# 04 — Analysis: from Gold to the two headline findings

This notebook computes nothing new. Everything here reads `gold.bid.*` tables
produced by `03_gold_bid_performance` and renders the two comparisons the
README leads with, plus the data-quality context that qualifies them.

Both this notebook and the README are generated from the same seeded run
(`--seed 42`), so the headline percentages should match. Where a figure here
involves a judgement call (e.g. which loss reasons count as price-related),
the classification is made explicit in the cell rather than assumed.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Aggregates only — every gold table here is small enough (a handful of
# rows per dimension) that a single-node toPandas() is the right call.
# Pulling the underlying fact table to the driver would not be.
overall            = spark.table("gold.bid.performance_overall").toPandas()
by_channel         = spark.table("gold.bid.performance_by_channel").toPandas()
by_executive       = spark.table("gold.bid.performance_by_account_executive").toPandas()
by_segment         = spark.table("gold.bid.performance_by_segment").toPandas()
by_value_band      = spark.table("gold.bid.performance_by_value_band").toPandas()
loss_coverage      = spark.table("gold.bid.loss_reason_coverage").toPandas()
loss_reasons       = spark.table("gold.bid.loss_reasons").toPandas()
open_pipeline      = spark.table("gold.bid.open_pipeline").toPandas()

## Finding 1 — the migration artefact

Bulk-loaded bids convert at roughly a third of the rate of organically
entered ones. The gap is not noise — it is 58% of the dataset behaving like
a different population, concentrated in one executive's portfolio.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel 1: bulk vs organic, company-wide -------------------------------
channel_labels = by_channel["is_bulk_load"].map({True: "Bulk-loaded", False: "Organic"})
axes[0].bar(channel_labels, by_channel["win_rate_by_count"],
            color=["#c0c0c0", "#2b6cb0"])
axes[0].set_title("Win rate: bulk vs organic")
axes[0].set_ylabel("Win rate")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_channel["win_rate_by_count"]):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center")

# --- Panel 2: Executive 4, all bids vs organic-only ------------------------
exec4 = by_executive[by_executive["account_executive"] == "Executive 4"].iloc[0]
company_avg = overall["win_rate_by_count"].iloc[0]

bars = axes[1].bar(
    ["All bids", "Organic only", "Company\naverage"],
    [exec4["wr_all"], exec4["wr_organic"], company_avg],
    color=["#c0392b", "#2b6cb0", "#888888"],
)
axes[1].set_title("Executive 4: worst performer, or artefact?")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, [exec4["wr_all"], exec4["wr_organic"], company_avg]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/finding1_migration_artefact.png", bbox_inches="tight")
plt.show()

print(f"Executive 4 — all bids: {exec4['wr_all']:.1f}%   organic only: {exec4['wr_organic']:.1f}%   "
      f"gap: {exec4['artefact_gap']:.1f}pp")

### Caveat this chart cannot show

The bulk pool was drawn concentrated in Financial Services / Public Sector —
the two lowest-baseline-win-rate segments in the dataset. Removing bulk load
moves Executive 4 from worst to above-average, but part of what's left could
still be segment mix, not a clean organic signal. The section right after
this one tests that directly with a regression that holds segment, state,
and deal size constant.

## Controlling for the confound: does the effect survive?

The chart above compares bulk-loaded bids to organic ones without accounting
for *which* bids ended up in each bucket. The generator concentrated bulk
records in Financial Services, Public Sector, and Executive 4's portfolio —
segments that convert below average regardless of how the bid was entered.
So the 16.8pp gap could be partly (or entirely) segment mix wearing a
migration-artefact costume.

A logistic regression that holds segment, state, and deal size constant
answers this directly: after adjusting for everything else in the row, does
`is_bulk_load` still move the odds of winning?

In [0]:
%pip install statsmodels -q

If the import below fails with a stale-module error, run
`dbutils.library.restartPython()` in a new cell and re-run the notebook from
the top — this is the first cell that imports `statsmodels`, so a restart
usually isn't needed, but Databricks' environment caching is inconsistent
about it.

In [0]:
import numpy as np
import statsmodels.formula.api as smf

# Row-level data, not the Gold aggregates — this is the one place in the
# notebook that needs it. Same join/filter as 03_gold_bid_performance's
# `fact`, kept independent here so this section doesn't depend on Gold
# re-exposing row-level output it was never meant to.
bids_clean = spark.table("silver.bid.bids_clean")
clients_clean = spark.table("silver.bid.clients_clean")

reg_pdf = (
    bids_clean.join(clients_clean, "client_id", "left")
    .filter("outcome IS NOT NULL")
    .select("outcome", "is_bulk_load", "segment", "state", "contract_value_brl")
    .toPandas()
)

# Continuous log-value instead of the Gold layer's quartile bins — a
# regression doesn't need binning, and log keeps the right-skewed value
# distribution from dominating the fit.
reg_pdf["log_value"] = np.log(reg_pdf["contract_value_brl"])

print(f"rows going into the model: {len(reg_pdf)}")

**Why segment and state are in the model, and account_executive isn't:**
segment and state are what the bulk pool was concentrated in — they're the
confound. `account_executive` is almost the same variable as `is_bulk_load`
here (the bulk batches sit almost entirely in Executive 4's portfolio by
construction), so including it would absorb the very effect this model is
trying to measure, not control for a separate one.

In [0]:
model = smf.logit(
    "outcome ~ C(is_bulk_load) + C(segment) + C(state) + log_value",
    data=reg_pdf,
).fit(disp=False)

BULK_TERM = "C(is_bulk_load)[T.True]"
coef = model.params[BULK_TERM]
ci_lo, ci_hi = model.conf_int().loc[BULK_TERM]
pval = model.pvalues[BULK_TERM]

adjusted_or = np.exp(coef)
print(f"Adjusted odds ratio for is_bulk_load: {adjusted_or:.2f}   "
      f"95% CI [{np.exp(ci_lo):.2f}, {np.exp(ci_hi):.2f}]   p={pval:.1e}")

# Naive odds ratio for comparison — the number Finding 1's chart implies
# without any controls.
p_bulk = reg_pdf.loc[reg_pdf.is_bulk_load, "outcome"].mean()
p_org = reg_pdf.loc[~reg_pdf.is_bulk_load, "outcome"].mean()
naive_or = (p_bulk / (1 - p_bulk)) / (p_org / (1 - p_org))
print(f"Naive odds ratio (no controls):       {naive_or:.2f}")

**Reading the comparison:** if the adjusted odds ratio had moved sharply
toward 1 relative to the naive one, that would mean segment/state/value mix
was doing most of the work and the "migration artefact" story was
overstated. If it stays close to the naive figure, the bulk-load effect is
real and not just a proxy for which segments got bulk-loaded — which is what
lets Finding 1's headline number stand without a segment-mix asterisk on it.

This still isn't causal proof — it's an observational adjustment for the
confounders the dataset makes obvious (segment, state, deal size), not every
confounder that could exist. `model.summary()` has the full coefficient
table if you want to see how segment and state individually move win
probability.

## Finding 2 — win rate by count vs by value

The company wins small contracts and loses large ones. Counting bids
flatters performance; weighting by revenue does not.

In [0]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(by_value_band["value_quartile"].astype(str), by_value_band["win_rate_by_count"],
       color="#2b6cb0")
ax.set_title("Win rate by contract-value quartile")
ax.set_xlabel("Value quartile (1 = smallest, 4 = largest)")
ax.set_ylabel("Win rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_value_band["win_rate_by_count"]):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/finding2_value_effect.png", bbox_inches="tight")
plt.show()

gap = overall["win_rate_by_count"].iloc[0] - overall["win_rate_by_value"].iloc[0]
print(f"Win rate by count: {overall['win_rate_by_count'].iloc[0]:.1f}%   "
      f"by value: {overall['win_rate_by_value'].iloc[0]:.1f}%   gap: {gap:.1f}pp")

## Why it happens: loss reasons, and how little of the picture they cover

The reason field would settle whether Finding 2 is a pricing problem. It's
populated for a single-digit share of losses — reported here as a signal,
explicitly not a population estimate.

In [0]:
coverage_pct = loss_coverage["coverage_pct"].iloc[0]
print(f"Loss reason coverage: {coverage_pct:.1f}% "
      f"({loss_coverage['losses_with_reason'].iloc[0]} of {loss_coverage['losses_total'].iloc[0]} losses)")
print(f"Attributed to the placeholder competitor: {loss_coverage['placeholder_attributed'].iloc[0]}")

# Explicit exclusion list rather than a keyword regex: with only a
# handful of distinct reasons, naming the non-price ones directly is more
# reliable than a substring match (e.g. "Financial proposal not
# competitive" is price-related but contains none of cost/price/commercial).
NON_PRICE_REASONS = {
    "Bid cancelled by client",
    "Scope did not match expectations",
    "Incumbent held established relationship",
    "Bid suspended / postponed",
    "Insufficient information from client",
    "Reason not recorded",
}
price_related = loss_reasons[~loss_reasons["loss_reason"].isin(NON_PRICE_REASONS)]
price_share = price_related["share_pct"].sum()
print(f"Price/cost-related share of recorded reasons: {price_share:.1f}%")

loss_reasons.sort_values("losses", ascending=False).head(10)

## Open pipeline

Reported on its own — never folded into the loss column, which would
understate the win rate.

In [0]:
open_pipeline

## What this notebook does not claim

See the README's "What this analysis cannot tell you" section for the full
list — sales-cycle length, the non-random loss-reason sample, monthly vs
total contract value, and the absence of bid cost. Repeating it here would
just be duplication; the point is that those limitations apply to every
chart above, not only to the text they sit next to.